# Minimal reproduction: a `required` slot on one class bleeds its `sh:minCount` onto every OTHER class sharing the same `class_uri`

Prepared as supporting evidence for
[linkml/linkml#3932](https://github.com/linkml/linkml/issues/3932). Found
while diagnosing Health-DCAT-AP-plus/ResHealth-DCAT-AP's own residual
SHACL violations (`KNOWN_MERGED_SHAPES_VIOLATIONS`' `value` entry --
dcat-ap-plus's `QualitativeAttribute` requires `prov:value`, and shares
`class_uri: prov:Entity` with `Entity`/`EvaluatedEntity`/
`AnalysisSourceData`).

**Not a new mechanism** -- the underlying "distinct classes sharing one
`class_uri` get merged into a single SHACL NodeShape" behavior is already
reported at [linkml/linkml#3011](https://github.com/linkml/linkml/issues/3011)
("Unintentional SHACL Class merge," closed, partially fixed by
[#3020](https://github.com/linkml/linkml/pull/3020)) -- filed by a
dcat-ap-plus contributor, using dcat-ap-plus's own schema as the example.
That issue's own example showed the merge producing duplicated
`sh:description` values -- a cosmetic symptom. **This notebook shows the
same merge causing a functional one instead**: a `required: true` slot on
only one of the two merged classes ends up constraining *every* instance
of the shared `class_uri`, including ones that never declared anything
about that slot -- so actually-valid data starts failing validation, not
just growing an extra description string.

**Zero dependency on dcat-ap-plus** -- two classes, one slot, no imports
beyond `linkml:types`. **Deliberately atomic**: exactly one triple, a
bare class-typing assertion, nothing else.

In [1]:
from pathlib import Path
import pyshacl
from linkml.generators.shaclgen import ShaclGenerator
from rdflib import Graph

SCHEMA = Path("minimal-class-uri-cardinality-bleed.yaml")
assert SCHEMA.exists()

## 1. Generate SHACL from the minimal schema

Two classes, `Entity` and `QualitativeAttribute`, **both** `class_uri:
ex:Entity` -- a real, deliberate LinkML pattern ("this is also, more
specifically, an Entity"), not a mistake. Only `QualitativeAttribute`
declares a slot, `value`, `required: true`.

In [2]:
shapes_ttl = ShaclGenerator(str(SCHEMA)).serialize()
print(shapes_ttl)

@prefix ex: <https://example.org/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix sh: <http://www.w3.org/ns/shacl#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

ex:Entity a sh:NodeShape ;
    rdfs:comment "A plain entity -- never declares anything about `value`, required or otherwise.",
        "Shares Entity's own class_uri deliberately (the same pattern dcat-ap-plus uses for Entity/EvaluatedEntity/AnalysisSourceData/ QualitativeAttribute, all class_uri: prov:Entity). Requires `value`." ;
    sh:closed true ;
    sh:ignoredProperties ( rdf:type ),
        ( rdf:type ) ;
    sh:property [ sh:datatype xsd:string ;
            sh:description "Required on QualitativeAttribute only -- but the generated SHACL constraint ends up on the single merged ex:Entity NodeShape, so it applies to every ex:Entity instance, not just QualitativeAttribute ones." ;
            sh:maxCount 1 ;
            sh:minCount 1 

There is exactly **one** `sh:NodeShape` here, `ex:Entity` -- `Entity`'s
own shape and `QualitativeAttribute`'s own shape have been merged into it,
since they share a `class_uri`. The merged shape carries `sh:minCount 1`
on `ex:value` -- a constraint that only `QualitativeAttribute` ever
declared.

## 2. The one-triple example -- a bare `Entity`, nothing else

In [3]:
data_ttl = """
@prefix ex: <https://example.org/> .

<https://example.org/x> a ex:Entity .
"""
data_graph = Graph()
data_graph.parse(data=data_ttl, format="turtle")
print(data_graph.serialize(format="turtle"))

@prefix ex: <https://example.org/> .

ex:x a ex:Entity .




This node is a plain `Entity`, not a `QualitativeAttribute` -- it was
never supposed to need a `value`.

## 3. Validate

In [4]:
shapes_graph = Graph()
shapes_graph.parse(data=shapes_ttl, format="turtle")

conforms, results_graph, results_text = pyshacl.validate(
    data_graph,
    shacl_graph=shapes_graph,
    data_graph_format="turtle",
    inference="none",
    advanced=True,
)
print(f"conforms = {conforms}")
print(results_text)

conforms = False
Validation Report
Conforms: False
Results (1):
Constraint Violation in MinCountConstraintComponent (http://www.w3.org/ns/shacl#MinCountConstraintComponent):
	Severity: sh:Violation
	Source Shape: [ sh:datatype xsd:string ; sh:description Literal("Required on QualitativeAttribute only -- but the generated SHACL constraint ends up on the single merged ex:Entity NodeShape, so it applies to every ex:Entity instance, not just QualitativeAttribute ones.") ; sh:maxCount Literal("1", datatype=xsd:integer) ; sh:minCount Literal("1", datatype=xsd:integer) ; sh:nodeKind sh:Literal ; sh:order Literal("0", datatype=xsd:integer) ; sh:path ex:value ]
	Focus Node: ex:x
	Result Path: ex:value
	Message: Less than 1 values on ex:x->ex:value



## What happened

`conforms = False` -- `MinCountConstraintComponent`, "Less than 1 values
on ex:x->ex:value". A bare `Entity`, which never declared anything about
`value`, fails validation for lacking one, purely because
`QualitativeAttribute` -- a *different* class that happens to share
`Entity`'s `class_uri` -- requires it.

**Root cause**: same mechanism as
[linkml/linkml#3011](https://github.com/linkml/linkml/issues/3011) --
`ShaclGenerator` keys generated NodeShapes by `class_uri`, so distinct
LinkML classes that deliberately share one (a legitimate, common pattern)
get folded into a single shape. #3011's own example showed this merging
`sh:description` values -- harmless duplication. This reproduction shows
the same merge folding `sh:minCount`/`sh:maxCount`/`sh:datatype`
constraints together too, which is not harmless: it silently narrows the
valid-data envelope of every class sharing that `class_uri` to the
*union* of all their slots' requirements, not just the ones each class
actually declared.